# 02 — Иерархическая кластеризация, DBSCAN и HDBSCAN

# Кластеризация данных: практическая серия ноутбуков

сначала формулируется идея метода, затем показывается его геометрический смысл, визуализация и практическая реализация в Python. В исходном уроке акцент сделан на объяснении метода через визуальные представления и двумерные проекции;

## 1. Иерархическая кластеризация

Иерархические методы строят структуру вложенных кластеров.

В агломеративном варианте сначала каждая точка считается отдельным кластером. Затем алгоритм последовательно объединяет наиболее близкие группы. Результат можно представить **дендрограммой**.

Ключевой параметр — способ измерения расстояния между кластерами (**linkage**):
- `ward` — минимизирует рост внутрикластерной дисперсии;
- `complete` — использует максимальное расстояние между объектами двух групп;
- `average` — среднее расстояние;
- `single` — минимальное расстояние.

### Преимущества
- не требуется заранее строить центроиды;
- можно увидеть иерархию объединений;
- полезна для исследования вложенной структуры.

### Недостатки
- вычислительно дороже на больших наборах;
- ранние объединения обычно нельзя «отменить»;
- результат зависит от linkage и метрики расстояния;
- дендрограмма может быть трудно читаема при большом `n`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mglearn

from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
X_moons, y_moons = make_moons(
    n_samples=500, noise=0.07, random_state=42
)
X_moons = StandardScaler().fit_transform(X_moons)

plt.figure(figsize=(8, 5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], s=20)
plt.title("Учебные данные: две лунообразные группы")
plt.xlabel("Признак 1")
plt.ylabel("Признак 2")
plt.show()

## 2. Агломеративная кластеризация на нелинейных данных

Для сравнения попробуем разные linkage. `ward` требует евклидовой геометрии и особенно естественен для компактных кластеров; `single` может «сцеплять» точки цепочкой.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, linkage_name in zip(axes, ["ward", "complete", "average"]):
    model = AgglomerativeClustering(
        n_clusters=2, linkage=linkage_name
    )
    labels = model.fit_predict(X_moons)
    mglearn.discrete_scatter(X_moons[:, 0], X_moons[:, 1], labels, ax=ax, s=18)
    ax.set_title(f"Linkage = {linkage_name}")
    ax.set_xlabel("Признак 1")
    ax.set_ylabel("Признак 2")

plt.tight_layout()
plt.show()

**Интерпретация:** иерархический подход гибче K-means в выборе формы кластеров, но конкретный результат сильно зависит от linkage. Для реальных данных параметр нельзя выбирать механически.

## 3. Дендрограмма

Дендрограмма показывает порядок объединения кластеров и расстояния, на которых эти объединения происходят. Горизонтальный «срез» дендрограммы задаёт количество групп.

Для визуальной демонстрации используем небольшую выборку: строить полную дендрограмму для десятков тысяч объектов нецелесообразно.

In [ ]:
rng = np.random.RandomState(42)
idx = rng.choice(len(X_moons), size=80, replace=False)
Z = linkage(X_moons[idx], method="average")

plt.figure(figsize=(13, 5))
dendrogram(Z, no_labels=True)
plt.title("Дендрограмма агломеративной кластеризации")
plt.xlabel("Объекты/подкластеры")
plt.ylabel("Расстояние объединения")
plt.show()

## 4. DBSCAN

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** строит кластеры на основе плотности.

Два главных параметра:
- `eps` — радиус окрестности;
- `min_samples` — минимальное число соседей, необходимое для плотной области.

Точки могут стать:
- **core** — имеют достаточно соседей;
- **border** — находятся рядом с core-точкой, но сами не достаточно плотные;
- **noise** — не относятся ни к одному кластеру.

### Преимущества
- не нужно задавать число кластеров;
- умеет находить кластеры сложной формы;
- естественно выделяет шум.

### Недостатки
- чувствителен к `eps` и `min_samples`;
- плохо работает при сильно различающейся плотности кластеров;
- масштаб признаков критичен;
- в многомерном пространстве выбор `eps` становится сложнее.

In [ ]:
db = DBSCAN(eps=0.25, min_samples=6)
db_labels = db.fit_predict(X_moons)

plt.figure(figsize=(8, 5))
mglearn.discrete_scatter(X_moons[:, 0], X_moons[:, 1], db_labels, ax=plt.gca(), s=20)
plt.title("DBSCAN на лунообразных данных (-1 = шум)")
plt.xlabel("Признак 1")
plt.ylabel("Признак 2")
plt.show()

print("Количество найденных кластеров:",
      len(set(db_labels)) - (1 if -1 in db_labels else 0))
print("Количество шумовых точек:", np.sum(db_labels == -1))

## 5. HDBSCAN

**HDBSCAN** можно рассматривать как развитие идеи DBSCAN: вместо одного фиксированного уровня плотности алгоритм исследует иерархию плотностей и выбирает устойчивые кластеры.

### Преимущества
- лучше приспособлен к разной плотности;
- не требует заранее задавать число кластеров;
- умеет выделять шум;
- параметр `min_cluster_size` часто проще интерпретировать, чем единственный `eps`.

### Недостатки
- параметров всё равно несколько;
- вычислительно сложнее;
- результат зависит от выбранной метрики и параметров;
- интерпретация при очень сложной структуре может быть нетривиальной.

В зависимости от версии scikit-learn HDBSCAN может быть доступен как `sklearn.cluster.HDBSCAN`. Иначе используется пакет `hdbscan`.

In [ ]:
# Вариант для новых версий scikit-learn:
try:
    from sklearn.cluster import HDBSCAN
    hdb = HDBSCAN(min_cluster_size=20)
    hdb_labels = hdb.fit_predict(X_moons)
    print("Использован sklearn.cluster.HDBSCAN")
except ImportError:
    # Если HDBSCAN отсутствует в вашей версии sklearn:
    # pip install hdbscan
    import hdbscan
    hdb = hdbscan.HDBSCAN(min_cluster_size=20)
    hdb_labels = hdb.fit_predict(X_moons)
    print("Использован внешний пакет hdbscan")

plt.figure(figsize=(8, 5))
mglearn.discrete_scatter(X_moons[:, 0], X_moons[:, 1], hdb_labels, ax=plt.gca(), s=20)
plt.title("HDBSCAN на лунообразных данных")
plt.xlabel("Признак 1")
plt.ylabel("Признак 2")
plt.show()

print("Количество кластеров:",
      len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0))
print("Шум:", np.sum(hdb_labels == -1))

## 6. Какой метод когда выбирать?

- **K-means** — компактные кластеры, большой объём данных, понятное `K`.
- **Иерархический** — нужна структура вложенных групп или дендрограмма.
- **DBSCAN** — сложная форма кластеров и наличие шума при примерно одинаковой плотности.
- **HDBSCAN** — сложная структура плотности и желание автоматически отделять шум.

## 7. Псевдоалгоритм DBSCAN

Ниже намеренно упрощённая реализация: она показывает логику поиска плотных областей, а не заменяет оптимизированную библиотечную реализацию.

In [ ]:
def dbscan_pseudocode(X, eps, min_samples):
    labels = np.full(len(X), -2)  # -2 = ещё не обработана
    cluster_id = 0

    for i in range(len(X)):
        if labels[i] != -2:
            continue

        neighbors = range_query(X, i, eps)

        if len(neighbors) < min_samples:
            labels[i] = -1  # шум
            continue

        labels[i] = cluster_id
        queue = list(neighbors)

        while queue:
            j = queue.pop()

            if labels[j] == -1:
                labels[j] = cluster_id

            if labels[j] != -2:
                continue

            labels[j] = cluster_id
            j_neighbors = range_query(X, j, eps)

            if len(j_neighbors) >= min_samples:
                queue.extend(j_neighbors)

        cluster_id += 1

    return labels

# range_query(X, i, eps) должна вернуть индексы точек
# на расстоянии <= eps от X[i].

## 8. Вывод

Главное различие методов — **какую геометрию кластера они считают естественной**. K-means ищет компактные группы вокруг центров, иерархические методы строят дерево объединений, а DBSCAN/HDBSCAN используют плотность и могут выделять шум.